# XGBoost Hyperparameter Trial Comparison

Run this notebook **after** the baseline and individual trial notebooks. It combines their saved validation results. The held-out test set is not used here.

In [ ]:
from pathlib import Path
import os
import pandas as pd

try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ.setdefault("INTRO_AI_PROJECT_ROOT", "/content/drive/MyDrive/Colab Notebooks/Education/INM701")
except Exception:
    pass


def resolve_project_root():
    candidates = []
    explicit = os.environ.get("INTRO_AI_PROJECT_ROOT")
    if explicit:
        candidates.append(Path(explicit).expanduser())
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    candidates.extend([
        Path(r"C:/Users/PC/Documents/GitHub/Intro-to-AI"),
        Path("/content/drive/MyDrive/Colab Notebooks/Education/INM701"),
    ])
    for candidate in candidates:
        if (candidate / "outputs" / "outputs" / "xgboost" / "tables").exists():
            return candidate
        if (candidate / "outputs" / "xgboost" / "tables").exists():
            return candidate
    return candidates[0]


def first_existing_path(candidates):
    expanded = [Path(candidate).expanduser() for candidate in candidates]
    for candidate in expanded:
        if candidate.exists():
            return candidate
    return expanded[0]


PROJECT_ROOT = resolve_project_root()
OUTPUTS_ROOT = first_existing_path([
    PROJECT_ROOT / "outputs" / "outputs",
    PROJECT_ROOT / "outputs",
])
print("PROJECT_ROOT:", PROJECT_ROOT)


In [ ]:
TABLES_DIR = OUTPUTS_ROOT / "xgboost" / "tables"
EXPECTED_FILES = ['xgboost_baseline.csv', 'xgboost_n_estimators_trial.csv', 'xgboost_max_depth_trial.csv', 'xgboost_learning_rate_trial.csv', 'xgboost_subsample_trial.csv', 'xgboost_colsample_bytree_trial.csv']

frames = []
missing = []
for filename in EXPECTED_FILES:
    path = TABLES_DIR / filename
    if path.exists():
        frame = pd.read_csv(path)
        frame["source_file"] = filename
        frames.append(frame)
    else:
        missing.append(filename)

if missing:
    print("Missing result files (run those notebooks first):")
    for item in missing:
        print(" -", item)

if not frames:
    raise FileNotFoundError("No trial result CSV files were found.")

all_trials_df = pd.concat(frames, ignore_index=True, sort=False)
all_trials_df = all_trials_df.sort_values("f1", ascending=False).reset_index(drop=True)
display(all_trials_df)

comparison_path = TABLES_DIR / "xgboost_all_individual_trials.csv"
all_trials_df.to_csv(comparison_path, index=False)
print("Saved combined comparison:", comparison_path)
print("Highest validation F1 row:")
display(all_trials_df.head(1))
